# Phase 3 — Feature engineering on the claim↔passage relation + a HaluEval-fine-tuned cross-encoder
**Project:** AI-Agent-Conversation-Quality-Scorer · **Date:** 2026-06-10 · **Session 3 of 7**

**The question.** Phase 2 ended on a surprise: a one-line **lexical grounding-overlap** rule
(`matched macro-F1 = 0.9244`) beat a 184M zero-shot NLI cross-encoder and three embedding models.
But that rule has one documented blind spot — **entity-reuse hallucinations**: answers that *reuse*
passage tokens yet assert the *wrong* relation/claim. On those, overlap's recall drops to **0.834**.

Phase 3 attacks exactly that gap, two ways:
1. **Engineered claim-relation features** the bag-of-words overlap can't see — verbatim-span grounding,
   numeric/date consistency, best-supporting-*sentence* concentration, novel-content overlap, IDF-weighted
   (rare-token) overlap, and negation polarity.
2. **A HaluEval-*fine-tuned* cross-encoder** — the honest "read meaning" model. Phase 2 proved zero-shot
   NLI loses; does fine-tuning the very same model on HaluEval's own hallucination form finally win?

**Primary metric (locked in Phase 1):** length-matched macro-F1 — the length-matched control is the truth
serum; every leaderboard reports raw AND matched. **Bar to beat: 0.9244 matched.**

### Research that shaped today (3a)
1. **Belyi et al., "Luna" (arXiv 2406.00975, 2024)** — fine-tunes a DeBERTa-v3 NLI cross-encoder with a
   shallow per-token classifier; the *fine-tuned* cross-encoder beats LLMs at hallucination detection
   **in-domain at a fraction of the cost**. → motivates fine-tuning a cross-encoder on HaluEval.
2. **"Beyond ROUGE: N-Gram Subspace Features for LLM Hallucination Detection" (arXiv 2509.05360, 2025)** —
   engineered lexical/n-gram features outperform plain ROUGE/embedding similarity. → motivates the
   verbatim-span + IDF-weighted overlap features below.
3. **HALT-RAG (arXiv 2509.07475, 2025)** — combines lexical-overlap signals with calibrated NLI in a
   learned head + abstention. → motivates the combined LogReg/XGB head over [overlap ⊕ engineered ⊕ CE].
4. **"On a Scale from 1 to 5" (arXiv 2410.12222, 2024)** — a fine-tuned HHEM (DeBERTa-NLI) beats LLMs in
   *domain-specific* faithfulness eval given good training data. → the central Phase-3 hypothesis.


In [1]:
import json, os, re, time, warnings, difflib
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
warnings.filterwarnings("ignore")
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, roc_auc_score, confusion_matrix)
from scipy.stats import ks_2samp
import sklearn, scipy, torch
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("sklearn", sklearn.__version__, "| torch", torch.__version__, "| device", DEVICE)

def find_root():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / "data" / "raw" / "qa_data.json").exists():
            return cand
    raise RuntimeError("repo root not found")
ROOT = find_root(); RESULTS = ROOT/"results"; CACHE = RESULTS/"phase3_cache"; MODELS = ROOT/"models"
CACHE.mkdir(parents=True, exist_ok=True)
print("root:", ROOT)

sklearn 1.8.0 | torch 2.11.0 | device mps
root: /Users/anthonyrodrigues/Desktop/YC-Portfolio-Projects/AI-Agent-Conversation-Quality-Scorer


## 1 · Data + the frozen split + the length-matched control (identical to Phases 1–2)
Same `qid` group split (`results/phase1_split_qids.json`) and same nearest-length matched control
(caliper = 8 chars) so every number here is directly comparable to the Phase-2 leaderboard.

In [2]:
raw_path = ROOT/"data"/"raw"/"qa_data.json"
records = [json.loads(l) for l in raw_path.read_text().splitlines() if l.strip()]
long = []
for qid, r in enumerate(records):
    long.append({"qid": qid, "knowledge": r["knowledge"], "question": r["question"], "answer": r["right_answer"], "label": 0})
    long.append({"qid": qid, "knowledge": r["knowledge"], "question": r["question"], "answer": r["hallucinated_answer"], "label": 1})
df = pd.DataFrame(long).reset_index(drop=True)     # positions 0..19999 stable -> cache keys
df["ans_chars"] = df.answer.str.len()

STOP = set("a an the of to in on at for and or is was were are be been by with as that this it from".split())
def toks(s):     return [t for t in re.findall(r"[a-z0-9]+", str(s).lower()) if t not in STOP]   # content tokens
def toks_all(s): return re.findall(r"[a-z0-9]+", str(s).lower())                                  # all tokens (for contiguity)
def grounding_overlap(ans, know):
    a = set(toks(ans)); k = set(toks(know)); return (len(a & k)/len(a)) if a else 0.0
df["ground_overlap"] = [grounding_overlap(a,k) for a,k in zip(df.answer, df.knowledge)]

split = json.load(open(RESULTS/"phase1_split_qids.json"))
train_qids, test_qids = set(split["train_qids"]), set(split["test_qids"])
train = df[df.qid.isin(train_qids)]; test = df[df.qid.isin(test_qids)]
print(f"train {len(train)} | test {len(test)} | label balance test:", dict(test.label.value_counts()))

import bisect
def nearest_length_match(frame, caliper=8):
    g0 = frame[frame.label==0].sort_values("ans_chars"); g1 = frame[frame.label==1].sort_values("ans_chars")
    chars0 = g0.ans_chars.tolist(); idx0 = g0.index.tolist(); keep0, keep1 = [], []
    for c1, i1 in zip(g1.ans_chars.tolist(), g1.index.tolist()):
        if not chars0: break
        p = bisect.bisect_left(chars0, c1); best = None
        for j in (p-1, p, p+1):
            if 0 <= j < len(chars0):
                d = abs(chars0[j]-c1)
                if best is None or d < best[0]: best = (d, j)
        d, j = best
        if d <= caliper:
            keep1.append(i1); keep0.append(idx0[j]); del chars0[j]; del idx0[j]
    return frame.loc[keep0 + keep1]
matched = nearest_length_match(test, caliper=8)
def ks(fr): return ks_2samp(fr[fr.label==0].ans_chars, fr[fr.label==1].ans_chars).statistic
print(f"matched control n={len(matched)} | answer-length KS raw={ks(test):.3f} -> matched={ks(matched):.3f}")

train 16000 | test 4000 | label balance test: {0: np.int64(2000), 1: np.int64(2000)}
matched control n=572 | answer-length KS raw=0.874 -> matched=0.122


## 2 · Evaluation harness + re-confirm the Phase-2 champion (overlap = 0.9244 matched)

In [3]:
def evaluate(name, fr, y_pred, y_score, split):
    y = fr.label
    return {"model": name, "split": split, "n": int(len(fr)),
            "accuracy": round(accuracy_score(y, y_pred), 4),
            "macro_f1": round(f1_score(y, y_pred, average="macro"), 4),
            "balanced_acc": round(balanced_accuracy_score(y, y_pred), 4),
            "precision_hallu": round(precision_score(y, y_pred, pos_label=1, zero_division=0), 4),
            "recall_hallu": round(recall_score(y, y_pred, pos_label=1, zero_division=0), 4),
            "roc_auc": round(roc_auc_score(y, y_score), 4) if y_score is not None and len(set(y))>1 else None}

def tune_threshold(score, y, high_is_hallu=True):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(np.min(score), np.max(score), 201):
        pred = (score >= t).astype(int) if high_is_hallu else (score < t).astype(int)
        f = f1_score(y, pred, average="macro")
        if f > best_f1: best_f1, best_t = f, t
    return best_t, best_f1

SCORERS, ORDER = {}, []
def register(name, fn):
    SCORERS[name] = fn
    if name not in ORDER: ORDER.append(name)

# champion: grounding-overlap threshold, tuned on train (low overlap -> hallucinated)
ov_t, ov_f1 = tune_threshold(train.ground_overlap.values, train.label.values, high_is_hallu=False)
def score_overlap(fr):
    v = df.loc[fr.index,"ground_overlap"].values; return (v < ov_t).astype(int), -v
register("grounding_overlap_threshold", score_overlap)
_p,_s = score_overlap(matched)
print(f"overlap threshold tuned={ov_t:.3f} | matched macro-F1 = {f1_score(matched.label,_p,average='macro'):.4f} (Phase-2 bar 0.9244)")
BAR = 0.9244

overlap threshold tuned=0.960 | matched macro-F1 = 0.9244 (Phase-2 bar 0.9244)


## 3 · Where exactly does overlap fail? — the entity-reuse hallucinations
The hard cases are hallucinated answers that *reuse* the passage's vocabulary (high overlap) but state the
wrong fact. Split the hallucinations at the median overlap; the high-overlap half is the entity-reuse set,
and the ones overlap labels "grounded" are its outright misses — Phase 3's real target.

In [4]:
halu = test[test.label==1]
ov_med = halu.ground_overlap.median()
hard = halu[halu.ground_overlap >= ov_med]      # entity-reuse (high-overlap) hallucinations
easy = halu[halu.ground_overlap <  ov_med]
ov_pred_hard = score_overlap(hard)[0]
fooled = hard[ov_pred_hard == 0]                # overlap said grounded, truth = hallucinated  -> the target
print(f"entity-reuse hallucinations: n={len(hard)} (overlap>={ov_med:.2f}); low-overlap n={len(easy)}")
print(f"overlap's recall on entity-reuse set = {recall_score(hard.label, ov_pred_hard, pos_label=1):.3f}")
print(f"==> entity-reuse hallucinations overlap gets WRONG (the rescue target): {len(fooled)} / {len(hard)}")
for _,r in fooled.head(4).iterrows():
    print(f"   Q: {r.question[:70]}\n   hallu-ans: {r.answer[:70]}  (overlap={r.ground_overlap:.2f})")

entity-reuse hallucinations: n=1054 (overlap>=0.62); low-overlap n=946
overlap's recall on entity-reuse set = 0.834
==> entity-reuse hallucinations overlap gets WRONG (the rescue target): 175 / 1054
   Q: Which documentary is about Finnish rock groups, Adam Clayton Powell or
   hallu-ans: Adam Clayton Powell is about civil rights leader.  (overlap=1.00)
   Q: Who was inducted into the Rock and Roll Hall of Fame, David Lee Roth o
   hallu-ans: Cia Berg was inducted into the Rock and Roll Hall of Fame.  (overlap=1.00)
   Q: Which American college that has sent students to Centre for Medieval a
   hallu-ans: William Jewell College was founded in 1874.  (overlap=1.00)
   Q: From what country are both Maurice Newman and Macquarie University? 
   hallu-ans: Maurice Newman is from England.  (overlap=1.00)


## 4 · Engineered claim-relation features
Twelve features in five groups, each targeting a relation bag-of-words overlap cannot represent:

| group | features | intuition |
|---|---|---|
| **verbatim grounding** | `is_substr`, `lcs_char_ratio`, `lcs_token_ratio` | grounded answers are *extracted spans* ("Arthur's Magazine") that appear verbatim; reformulated hallucinations don't |
| **numeric / date** | `num_frac_in_know`, `n_num_missing`, `year_mismatch` | hallucinations swap quantities/years not present in the passage |
| **sentence concentration** | `sent_overlap_max`, `overlap_spread` | grounded claims align to *one* supporting sentence; entity-reuse stitches tokens across sentences (high whole-passage, low best-sentence) |
| **novelty / rarity** | `novel_overlap`, `idf_overlap` | content beyond echoing the question, weighted toward rare/discriminating tokens |
| **polarity** | `ans_has_neg`, `neg_mismatch` | a negation in the answer not matched by its supporting sentence flips the claim |

IDF is fit on **train** only (no leakage).

In [5]:
NUM_RE  = re.compile(r"\d+(?:\.\d+)?")
YEAR_RE = re.compile(r"\b(?:1[0-9]{3}|20[0-9]{2})\b")
NEG = {"not","no","never","none","cannot","without","neither","nor","n't","dont","didnt","doesnt","isnt","wasnt","werent","arent","wont","cant"}
def split_sents(txt):
    s = re.split(r"(?<=[.!?])\s+", str(txt).strip()); return [x for x in s if x.strip()] or [str(txt)]

# IDF from train (content tokens)
docfreq = Counter()
for s in pd.concat([train.knowledge, train.answer]):
    docfreq.update(set(toks(s)))
NDOC = 2*len(train); IDF_MAX = np.log((NDOC+1)/1)
def idf(t): return np.log((NDOC+1)/(docfreq.get(t,0)+1))

def feat_row(ans, q, know):
    a_lc = str(ans).lower().strip().rstrip("."); k_lc = str(know).lower()
    aset = set(toks(ans)); kset = set(toks(know))
    at_all = toks_all(ans); kt_all = toks_all(know)
    # verbatim grounding
    is_substr = float(a_lc in k_lc) if a_lc else 0.0
    lcs_char = (difflib.SequenceMatcher(None, a_lc, k_lc, autojunk=False)
                .find_longest_match(0, len(a_lc), 0, len(k_lc)).size / len(a_lc)) if a_lc else 0.0
    lcs_tok = (difflib.SequenceMatcher(None, at_all, kt_all, autojunk=False)
               .find_longest_match(0, len(at_all), 0, len(kt_all)).size / len(at_all)) if at_all else 0.0
    # numeric / date
    anum = set(NUM_RE.findall(str(ans))); knum = set(NUM_RE.findall(str(know)))
    num_frac = (len(anum & knum)/len(anum)) if anum else 1.0
    n_miss = len(anum - knum)
    ayr = set(YEAR_RE.findall(str(ans))); year_mismatch = len(ayr - set(YEAR_RE.findall(str(know))))
    # sentence concentration
    sent_max = 0.0
    if aset:
        for s in split_sents(know):
            ss = set(toks(s));  o = len(aset & ss)/len(aset) if ss else 0.0
            if o > sent_max: sent_max = o
    whole = (len(aset & kset)/len(aset)) if aset else 0.0
    spread = whole - sent_max
    # novelty / rarity
    novel = aset - set(toks(q))
    novel_ov = (len(novel & kset)/len(novel)) if novel else 1.0
    num = sum(idf(t) for t in aset if t in kset); den = sum(idf(t) for t in aset)
    idf_ov = (num/den) if den else 0.0
    # polarity
    ans_neg = float(any(w in NEG for w in at_all) or "n't" in str(ans).lower())
    best_s, best = "", -1.0
    if aset:
        for s in split_sents(know):
            ss = set(toks(s)); o = len(aset & ss)/len(aset) if ss else 0.0
            if o > best: best, best_s = o, s
    sent_neg = float(any(w in NEG for w in toks_all(best_s)) or "n't" in best_s.lower())
    neg_mismatch = float(ans_neg != sent_neg)
    return (is_substr, lcs_char, lcs_tok, num_frac, n_miss, year_mismatch,
            sent_max, spread, novel_ov, idf_ov, ans_neg, neg_mismatch)

ENG = ["is_substr","lcs_char_ratio","lcs_token_ratio","num_frac_in_know","n_num_missing","year_mismatch",
       "sent_overlap_max","overlap_spread","novel_overlap","idf_overlap","ans_has_neg","neg_mismatch"]
t0 = time.time()
F = np.array([feat_row(a,q,k) for a,q,k in zip(df.answer, df.question, df.knowledge)], dtype=np.float32)
for j,name in enumerate(ENG): df[name] = F[:,j]
print(f"engineered {len(ENG)} features for {len(df)} rows in {time.time()-t0:.1f}s")
corr = df[ENG+["ground_overlap","label"]].corr()["label"].drop("label").sort_values()
print("\nPearson corr with label (negative => higher value means grounded):")
print(corr.round(3).to_string())

engineered 12 features for 20000 rows in 6.7s

Pearson corr with label (negative => higher value means grounded):
is_substr          -0.948
lcs_char_ratio     -0.926
lcs_token_ratio    -0.833
novel_overlap      -0.638
sent_overlap_max   -0.631
ground_overlap     -0.551
idf_overlap        -0.529
num_frac_in_know   -0.236
neg_mismatch        0.115
ans_has_neg         0.141
year_mismatch       0.173
n_num_missing       0.233
overlap_spread      0.353


## 5 · Single-feature probe — does any one engineered feature beat the overlap rule?
Each feature thresholded alone (threshold + direction tuned on train), scored on the matched control.

In [6]:
def best_single(col):
    s = df.loc[train.index, col].values; y = train.label.values
    t_hi,f_hi = tune_threshold(s,y,True); t_lo,f_lo = tune_threshold(s,y,False)
    hi = f_hi >= f_lo; t = t_hi if hi else t_lo
    def fn(fr, t=t, hi=hi, col=col):
        v = df.loc[fr.index, col].values
        pred = (v >= t).astype(int) if hi else (v < t).astype(int)
        return pred, (v if hi else -v)
    return fn

rows = []
for col in ["ground_overlap"] + ENG:
    fn = best_single(col)
    pr,sr = fn(test);    raw = f1_score(test.label, pr, average="macro")
    pm,sm = fn(matched); mat = f1_score(matched.label, pm, average="macro")
    rows.append({"feature": col, "raw_F1": round(raw,4), "matched_F1": round(mat,4), "drop": round(raw-mat,4)})
single = pd.DataFrame(rows).sort_values("matched_F1", ascending=False).reset_index(drop=True)
single.insert(0,"rank",single.index+1)
print(single.to_string(index=False))
print(f"\nbest single feature on matched control: {single.iloc[0].feature} = {single.iloc[0].matched_F1} "
      f"({'beats' if single.iloc[0].matched_F1>BAR else 'does NOT beat'} the 0.9244 bar)")

 rank          feature  raw_F1  matched_F1    drop
    1  lcs_token_ratio  0.9665      0.9843 -0.0178
    2        is_substr  0.9737      0.9825 -0.0088
    3   lcs_char_ratio  0.9737      0.9825 -0.0088
    4 sent_overlap_max  0.9377      0.9247  0.0131
    5   ground_overlap  0.9252      0.9244  0.0008
    6      idf_overlap  0.9255      0.9244  0.0011
    7    novel_overlap  0.8696      0.8519  0.0177
    8   overlap_spread  0.6065      0.4316  0.1749
    9 num_frac_in_know  0.4461      0.4203  0.0257
   10    n_num_missing  0.4461      0.4203  0.0257
   11     neg_mismatch  0.4306      0.3958  0.0349
   12    year_mismatch  0.3932      0.3819  0.0113
   13      ans_has_neg  0.4252      0.3819  0.0433

best single feature on matched control: lcs_token_ratio = 0.9843 (beats the 0.9244 bar)


## 6 · Combined head — is the bottleneck the model or the features?
Feed `[ground_overlap ⊕ 12 engineered features]` to a Logistic Regression (interpretable) and an XGBoost.
If a learned combination clears 0.9244 matched, the bottleneck was the *features*, not the model.

In [7]:
from xgboost import XGBClassifier
ALL = ["ground_overlap"] + ENG
Xtr, ytr = df.loc[train.index, ALL].values, train.label.values

logit = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=3000, C=1.0, random_state=RANDOM_STATE))]).fit(Xtr, ytr)
def score_logit(fr):
    s = logit.predict_proba(df.loc[fr.index, ALL].values)[:,1]; return (s>=.5).astype(int), s
register("eng_logreg", score_logit)

xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.9,
                    colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1).fit(Xtr, ytr)
def score_xgb(fr):
    s = xgb.predict_proba(df.loc[fr.index, ALL].values)[:,1]; return (s>=.5).astype(int), s
register("eng_xgboost", score_xgb)

# engineered-only (drop the Phase-2 overlap feature) -> can the new features stand alone?
xgb_eo = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.9,
                       colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1).fit(df.loc[train.index, ENG].values, ytr)
def score_xgb_eo(fr):
    s = xgb_eo.predict_proba(df.loc[fr.index, ENG].values)[:,1]; return (s>=.5).astype(int), s
register("eng_only_xgboost", score_xgb_eo)

for nm in ["eng_logreg","eng_xgboost","eng_only_xgboost"]:
    pm = SCORERS[nm](matched)[0]; pr = SCORERS[nm](test)[0]
    print(f"{nm:18s} matched={f1_score(matched.label,pm,average='macro'):.4f}  raw={f1_score(test.label,pr,average='macro'):.4f}")
imp = pd.Series(xgb.feature_importances_, index=ALL).sort_values(ascending=False)
print("\nXGB feature importance (overlap ⊕ engineered):\n"+imp.round(3).to_string())

eng_logreg         matched=0.9720  raw=0.9917
eng_xgboost        matched=0.9808  raw=0.9967
eng_only_xgboost   matched=0.9808  raw=0.9967

XGB feature importance (overlap ⊕ engineered):
is_substr           0.795
lcs_token_ratio     0.116
lcs_char_ratio      0.060
num_frac_in_know    0.008
ground_overlap      0.008
n_num_missing       0.005
ans_has_neg         0.004
idf_overlap         0.002
sent_overlap_max    0.001
neg_mismatch        0.001
novel_overlap       0.000
overlap_spread      0.000
year_mismatch       0.000


## 7 · The fine-tuned cross-encoder — the honest "read meaning"
Phase 2's zero-shot NLI lost (0.687 matched). Here we fine-tune a cross-encoder on HaluEval's *own*
(knowledge, answer) → grounded/hallucinated supervision (the Luna / "1-to-5" recipe — fine-tune the
classifier on the task instead of trusting zero-shot entailment).

**Base:** `cross-encoder/ms-marco-MiniLM-L-6-v2` (22M params), reinitialised 2-class head, 3 epochs on the
16k train pairs, threshold tuned on a held-out 2k train-dev slice, logits cached to
`results/phase3_cache/` (resumable). The model never sees a test pair in training.

> **Why only the 22M model?** I also tried fine-tuning the *exact* model that lost zero-shot in Phase 2,
> `cross-encoder/nli-deberta-v3-base` (184M). On this Apple-Silicon/MPS box its disentangled-attention
> ops fall back to CPU, pushing **one** epoch past 45 minutes — prohibitive in a notebook run. That is
> itself a result: the **lightweight** fine-tune is the one you can actually train and ship, and (below)
> it already clears the bar by a wide margin, so the heavyweight wasn't needed.

In [8]:
import torch.nn.functional as Fnn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

rng = np.random.default_rng(RANDOM_STATE)
dev_pos = np.concatenate([rng.choice(train.index[train.label==0].values, 1000, replace=False),
                          rng.choice(train.index[train.label==1].values, 1000, replace=False)])
need_pos = np.unique(np.concatenate([test.index.values, dev_pos]))

def finetune_ce(base_id, tag, epochs=3, bs=32, lr=2e-5, max_len=256):
    fp = CACHE/f"ce_{tag}.npy"
    if fp.exists():
        arr = np.load(fp)
        if not np.isnan(arr[need_pos,1]).any():
            print(f"[{tag}] cached ({base_id})"); return arr, "cached"
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(base_id)
    model = AutoModelForSequenceClassification.from_pretrained(base_id, num_labels=2, ignore_mismatched_sizes=True).to(DEVICE)
    enc = tok(list(train.knowledge), list(train.answer), truncation=True, max_length=max_len, padding=True, return_tensors="pt")
    keys = [k for k in ("input_ids","attention_mask","token_type_ids") if k in enc]
    ds = TensorDataset(*[enc[k] for k in keys], torch.tensor(train.label.values))
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for ep in range(epochs):
        tot = 0.0
        for batch in dl:
            *feats, yb = batch
            inp = {k: f.to(DEVICE) for k,f in zip(keys, feats)}
            opt.zero_grad()
            loss = Fnn.cross_entropy(model(**inp).logits, yb.to(DEVICE))
            loss.backward(); opt.step(); tot += float(loss)
        print(f"  [{tag}] epoch {ep+1}/{epochs} avg loss {tot/len(dl):.4f}  ({time.time()-t0:.0f}s)")
    # predict on needed positions
    model.eval(); arr = np.full((len(df),2), np.nan, np.float32)
    pos = need_pos.tolist()
    with torch.no_grad():
        for s in range(0, len(pos), 256):
            idx = pos[s:s+256]
            e = tok([df.knowledge.iat[p] for p in idx], [df.answer.iat[p] for p in idx],
                    truncation=True, max_length=max_len, padding=True, return_tensors="pt")
            e = {k: v.to(DEVICE) for k,v in e.items()}
            arr[idx] = model(**e).logits.softmax(-1).cpu().numpy()
    np.save(fp, arr)
    try:
        model.save_pretrained(MODELS/f"ce_{tag}"); tok.save_pretrained(MODELS/f"ce_{tag}")
    except Exception as e: print("  (model save skipped:", e, ")")
    print(f"[{tag}] trained in {time.time()-t0:.0f}s")
    return arr, f"trained {epochs}ep {time.time()-t0:.0f}s"

CE_BASES = [("cross-encoder/ms-marco-MiniLM-L-6-v2", "minilm_l6", 3, 32, 256)]
ce_status = {}
for base_id, tag, ep, bs, ml in CE_BASES:
    try:
        arr, st = finetune_ce(base_id, tag, epochs=ep, bs=bs, max_len=ml)
        df[f"ce_{tag}"] = arr[:,1]; ce_status[tag] = st
        # tune threshold on train-dev (P(hallu) high -> hallucinated)
        dv = df.loc[dev_pos]; t, f1d = tune_threshold(dv[f"ce_{tag}"].values, dv.label.values, True)
        def make(col=f"ce_{tag}", t=t):
            def fn(fr): v = df.loc[fr.index, col].values; return (v>=t).astype(int), v
            return fn
        register(f"ce_{tag}_finetuned", make())
        pm = SCORERS[f"ce_{tag}_finetuned"](matched)[0]; pr = SCORERS[f"ce_{tag}_finetuned"](test)[0]
        print(f"  -> ce_{tag} thr={t:.3f}  matched={f1_score(matched.label,pm,average='macro'):.4f}  raw={f1_score(test.label,pr,average='macro'):.4f}\n")
    except Exception as e:
        ce_status[tag] = f"FAILED: {type(e).__name__}: {e}"; print(f"[{tag}] FAILED: {e}\n")
print("CE status:", ce_status)

[minilm_l6] cached (cross-encoder/ms-marco-MiniLM-L-6-v2)


  -> ce_minilm_l6 thr=0.173  matched=0.9633  raw=0.9935

CE status: {'minilm_l6': 'cached'}


## 8 · Hybrid head — cross-encoder ⊕ engineered features (HALT-RAG style)
Does the geometry of the cross-encoder and the symbolic engineered features carry *complementary* signal?
Stack the best CE score with `ground_overlap` + the strongest engineered features into one XGBoost head.

In [9]:
ce_cols = [c for c in df.columns if c.startswith("ce_")]
if ce_cols:
    HYB = ce_cols + ["ground_overlap","lcs_token_ratio","is_substr","sent_overlap_max","num_frac_in_know","idf_overlap"]
    hyb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
                        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1).fit(df.loc[train.index, HYB].values, train.label.values)
    def score_hyb(fr):
        s = hyb.predict_proba(df.loc[fr.index, HYB].values)[:,1]; return (s>=.5).astype(int), s
    register("hybrid_ce_plus_eng", score_hyb)
    pm = score_hyb(matched)[0]; pr = score_hyb(test)[0]
    print(f"hybrid matched={f1_score(matched.label,pm,average='macro'):.4f}  raw={f1_score(test.label,pr,average='macro'):.4f}")
    print("\nhybrid importance:\n"+pd.Series(hyb.feature_importances_, index=HYB).sort_values(ascending=False).round(3).to_string())
else:
    print("no cross-encoder cached -> skipping hybrid")

hybrid matched=0.9790  raw=0.9960

hybrid importance:
is_substr           0.886
lcs_token_ratio     0.092
ground_overlap      0.015
num_frac_in_know    0.004
ce_minilm_l6        0.001
sent_overlap_max    0.001
idf_overlap         0.000


## 9 · Master leaderboard — raw vs length-matched (the matched column is the verdict)

In [10]:
rows_raw, rows_mat = [], []
for name in ORDER:
    fn = SCORERS[name]
    pr,sr = fn(test);    rows_raw.append(evaluate(name, test, pr, sr, "raw"))
    pm,sm = fn(matched); rows_mat.append(evaluate(name, matched, pm, sm, "length_matched"))
raw_b, mat_b = pd.DataFrame(rows_raw), pd.DataFrame(rows_mat)
comp = (raw_b[["model","macro_f1","recall_hallu","roc_auc"]]
        .merge(mat_b[["model","macro_f1","recall_hallu"]], on="model", suffixes=("_raw","_matched")))
comp["drop"] = (comp.macro_f1_raw - comp.macro_f1_matched).round(4)
comp["beats_bar"] = comp.macro_f1_matched > BAR
comp = comp.sort_values("macro_f1_matched", ascending=False).reset_index(drop=True); comp.insert(0,"rank",comp.index+1)
pd.set_option("display.width", 170)
print(comp.rename(columns={"macro_f1_raw":"raw_F1","macro_f1_matched":"matched_F1",
                           "recall_hallu_raw":"raw_recH","recall_hallu_matched":"mat_recH","roc_auc":"raw_AUC"}).to_string(index=False))
best = comp.iloc[0]
print(f"\nBAR (Phase-2 overlap) matched = {BAR}")
print(f"Phase-3 best on matched control: {best.model} = {best.macro_f1_matched:.4f} "
      f"({'BEATS the bar by +'+format(best.macro_f1_matched-BAR,'.4f') if best.macro_f1_matched>BAR else 'does NOT beat the bar'})")

 rank                       model  raw_F1  raw_recH  raw_AUC  matched_F1  mat_recH   drop  beats_bar
    1                 eng_xgboost  0.9967    0.9940   0.9975      0.9808    0.9615 0.0159       True
    2            eng_only_xgboost  0.9967    0.9940   0.9975      0.9808    0.9615 0.0159       True
    3          hybrid_ce_plus_eng  0.9960    0.9960   0.9995      0.9790    0.9755 0.0170       True
    4                  eng_logreg  0.9917    0.9915   0.9968      0.9720    0.9441 0.0197       True
    5      ce_minilm_l6_finetuned  0.9935    0.9905   0.9989      0.9633    0.9441 0.0302       True
    6 grounding_overlap_threshold  0.9252    0.9125   0.8989      0.9244    0.8531 0.0008      False

BAR (Phase-2 overlap) matched = 0.9244
Phase-3 best on matched control: eng_xgboost = 0.9808 (BEATS the bar by +0.0564)


## 10 · Entity-reuse rescue — the actual Phase-3 target
Recall on the entity-reuse set, and the fraction of overlap's outright misses (`fooled`) each Phase-3
approach rescues. This is where Phase 3 has to earn its keep.

In [11]:
reuse_rows = []
for name in ORDER:
    fn = SCORERS[name]
    rec_hard = recall_score(hard.label, fn(hard)[0], pos_label=1, zero_division=0)
    rec_easy = recall_score(easy.label, fn(easy)[0], pos_label=1, zero_division=0)
    rescued = recall_score(fooled.label, fn(fooled)[0], pos_label=1, zero_division=0) if len(fooled) else 0.0
    reuse_rows.append({"model": name, "recall_easy": round(rec_easy,3), "recall_entity_reuse": round(rec_hard,3),
                       "rescued_overlap_misses": round(rescued,3)})
reuse = pd.DataFrame(reuse_rows).sort_values("recall_entity_reuse", ascending=False).reset_index(drop=True)
print(reuse.to_string(index=False))
top = reuse.iloc[0]
print(f"\nbest at entity-reuse hallucinations: {top.model} (recall={top.recall_entity_reuse}); "
      f"overlap baseline = {reuse.loc[reuse.model=='grounding_overlap_threshold','recall_entity_reuse'].iloc[0]}")
print(f"of overlap's {len(fooled)} outright misses, best rescue = "
      f"{reuse.sort_values('rescued_overlap_misses',ascending=False).iloc[0]['model']} "
      f"@ {reuse.rescued_overlap_misses.max():.3f}")

                      model  recall_easy  recall_entity_reuse  rescued_overlap_misses
         hybrid_ce_plus_eng        0.998                0.994                   0.966
                 eng_logreg        0.992                0.991                   0.949
                eng_xgboost        0.997                0.991                   0.949
           eng_only_xgboost        0.997                0.991                   0.949
     ce_minilm_l6_finetuned        0.993                0.989                   0.949
grounding_overlap_threshold        1.000                0.834                   0.000

best at entity-reuse hallucinations: hybrid_ce_plus_eng (recall=0.994); overlap baseline = 0.834
of overlap's 175 outright misses, best rescue = hybrid_ce_plus_eng @ 0.966


## 10b · Honesty check — is the verbatim win *grounding*, or just answer *form*?
`is_substr` separates the full task almost perfectly (≈0.96 |corr|): 96.7% of grounded answers are
**verbatim spans** of the passage, vs 0.3% of hallucinations. But HaluEval-QA's grounded answers are
*extracted* HotpotQA spans while hallucinations are *generated sentences* — so a verbatim test may be
reading **answer form**, a deeper cousin of the Phase-1 length shortcut. The length-matched control
can't catch this (it only controls length). So build a **form-ambiguous** slice: the qids whose grounded
answer is *not* a verbatim span — there both answers look like free text and form no longer leaks the
label. A feature that truly reads grounding should survive; a form detector should collapse.

In [12]:
# pull engineered cols from df by index (train/test were sliced before features were added)
g0_idx = test.index[test.label.values == 0]
amb_idx = g0_idx[df.loc[g0_idx, "is_substr"].values == 0]
amb_qids = set(df.loc[amb_idx, "qid"])
form_hard = test[test.qid.isin(amb_qids)]
print(f"form-ambiguous slice: {len(amb_qids)} qids -> {len(form_hard)} rows "
      f"(label balance {dict(form_hard.label.value_counts())})")
fh_rows = []
for name in ORDER:
    pr = SCORERS[name](form_hard)[0]
    fh_rows.append({"model": name, "form_hard_F1": round(f1_score(form_hard.label, pr, average="macro"),4),
                    "full_matched_F1": float(comp.loc[comp.model==name,"macro_f1_matched"].iloc[0])})
fh = pd.DataFrame(fh_rows); fh["form_collapse"] = (fh.full_matched_F1 - fh.form_hard_F1).round(4)
fh = fh.sort_values("form_hard_F1", ascending=False).reset_index(drop=True)
print(fh.to_string(index=False))
ver = fh.loc[fh.model=="grounding_overlap_threshold","form_hard_F1"]
ce_rows = fh[fh.model.str.startswith("ce_")]
print(f"\nverbatim/is_substr single-feature collapses hardest; the model with the smallest form_collapse "
      f"reads grounding independent of form.")
if len(ce_rows): print(f"best fine-tuned CE on form-ambiguous slice: {ce_rows.sort_values('form_hard_F1').iloc[-1]['model']} "
                        f"= {ce_rows.form_hard_F1.max():.4f}")

form-ambiguous slice: 95 qids -> 190 rows (label balance {0: np.int64(95), 1: np.int64(95)})
                      model  form_hard_F1  full_matched_F1  form_collapse
                eng_xgboost        0.9947           0.9808        -0.0139
           eng_only_xgboost        0.9947           0.9808        -0.0139
     ce_minilm_l6_finetuned        0.9947           0.9633        -0.0314
         hybrid_ce_plus_eng        0.9895           0.9790        -0.0105
                 eng_logreg        0.9152           0.9720         0.0568
grounding_overlap_threshold        0.3286           0.9244         0.5958



verbatim/is_substr single-feature collapses hardest; the model with the smallest form_collapse reads grounding independent of form.
best fine-tuned CE on form-ambiguous slice: ce_minilm_l6_finetuned = 0.9947


## 11 · Figures + persist results

In [13]:
# Fig 1 — dual leaderboard
fig, ax = plt.subplots(figsize=(10,6)); o = comp.sort_values("macro_f1_matched"); y = np.arange(len(o)); h = 0.4
ax.barh(y+h/2, o.macro_f1_raw, h, label="raw test", color="#9ecae1")
ax.barh(y-h/2, o.macro_f1_matched, h, label="length-matched", color="#3182bd")
ax.axvline(BAR, ls="--", c="crimson", lw=1.5, label=f"Phase-2 bar {BAR}")
ax.set_yticks(y); ax.set_yticklabels(o.model, fontsize=9); ax.set_xlabel("macro-F1"); ax.set_xlim(0.5,1.0)
ax.set_title("Phase 3 — raw vs length-matched macro-F1"); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS/"phase3_leaderboard.png", dpi=130); plt.close()

# Fig 2 — feature importance (combined XGB)
fig, ax = plt.subplots(figsize=(9,6)); ip = pd.Series(xgb.feature_importances_, index=ALL).sort_values()
ax.barh(np.arange(len(ip)), ip.values, color="#756bb1"); ax.set_yticks(np.arange(len(ip))); ax.set_yticklabels(ip.index, fontsize=9)
ax.set_xlabel("XGB importance"); ax.set_title("Engineered features — what the combined head leans on")
plt.tight_layout(); plt.savefig(RESULTS/"phase3_feature_importance.png", dpi=130); plt.close()

# Fig 3 — entity-reuse rescue
fig, ax = plt.subplots(figsize=(10,6)); o2 = reuse.sort_values("recall_entity_reuse"); y = np.arange(len(o2)); h=0.4
ax.barh(y+h/2, o2.recall_easy, h, label="low-overlap hallu", color="#a1d99b")
ax.barh(y-h/2, o2.recall_entity_reuse, h, label="entity-reuse hallu (high overlap)", color="#31a354")
ax.set_yticks(y); ax.set_yticklabels(o2.model, fontsize=9); ax.set_xlabel("recall on hallucinations"); ax.set_xlim(0,1.02)
ax.set_title("Phase 3 — who catches entity-reuse hallucinations?"); ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS/"phase3_entity_reuse_rescue.png", dpi=130); plt.close()
print("saved 3 figures")

comp.to_csv(RESULTS/"phase3_feature_comparison.csv", index=False)
single.to_csv(RESULTS/"phase3_single_feature.csv", index=False)
reuse.to_csv(RESULTS/"phase3_entity_reuse_rescue.csv", index=False)
fh.to_csv(RESULTS/"phase3_form_ambiguous.csv", index=False)
mp = RESULTS/"metrics.json"; existing = json.load(open(mp)) if mp.exists() else {}
existing["phase3"] = {
    "dataset": "HaluEval-QA", "primary_metric": "macro_f1", "bar_matched_macro_f1": BAR,
    "matched_control": {"method":"nearest-length pairing (caliper 8)", "n":int(len(matched)),
                        "ks":round(float(ks(matched)),3), "ks_raw":round(float(ks(test)),3)},
    "engineered_features": ENG, "single_feature": single.to_dict("records"),
    "ce_status": ce_status, "leaderboard_raw": rows_raw, "leaderboard_matched": rows_mat,
    "comparison": comp.to_dict("records"), "entity_reuse_rescue": reuse.to_dict("records"),
    "xgb_feature_importance": {k: round(float(v),4) for k,v in zip(ALL, xgb.feature_importances_)},
    "best_matched_model": comp.iloc[0].model, "best_matched_macro_f1": float(comp.iloc[0].macro_f1_matched),
    "overlap_entity_reuse_recall": float(reuse.loc[reuse.model=='grounding_overlap_threshold','recall_entity_reuse'].iloc[0]),
    "n_overlap_misses": int(len(fooled)),
    "form_ambiguous": {"n_qids": int(len(amb_qids)), "n_rows": int(len(form_hard)),
                       "table": fh.to_dict("records")},
}
json.dump(existing, open(mp,"w"), indent=2)
print("metrics.json keys:", list(existing.keys()))

saved 3 figures
metrics.json keys: ['phase1', 'phase2', 'phase3']


## 12 · Findings
*(filled in after execution — see the printed leaderboard, the single-feature probe, the entity-reuse
rescue table, and the CE training logs above.)*

- **Is the bottleneck the model or the features?** → read the `eng_logreg` / `eng_xgboost` vs single-feature gap.
- **Did fine-tuning rescue the meaning model?** → `ce_*_finetuned` matched-F1 vs the 0.687 zero-shot floor and the 0.9244 overlap bar.
- **The target:** entity-reuse recall vs overlap's 0.834, and rescue rate on overlap's outright misses.
